In [1]:
import feedparser
import requests
import re
url = "https://www.cert.ssi.gouv.fr/feed/"
rss_feed = feedparser.parse(url)
for entry in rss_feed.entries:
    print("Titre :", entry.title)
    print("Description:", entry.description)
    print("Lien :", entry.link)
    print("Date :", entry.published)

Titre : [Màj] Vulnérabilité dans Microsoft Exchange Server (15 mai 2026)
Description: [Mise à jour du 11 juin 2026] Le 9 juin 2026, Microsoft a publié des versions correctives. [Publication initiale] Le 14 mai 2026, Microsoft a publié un avis de sécurité concernant la vulnérabilité CVE-2026-42897 affectant Exchange Server. Elle permet à un attaquant non authentifié de provoquer...
Lien : https://www.cert.ssi.gouv.fr/alerte/CERTFR-2026-ALE-005/
Date : Fri, 15 May 2026 00:00:00 +0000
Titre : Vulnérabilité dans Cisco Catalyst SD-WAN (05 juin 2026)
Description: Une vulnérabilité a été découverte dans Cisco Catalyst SD-WAN. Elle permet à un attaquant de provoquer une élévation de privilèges. Cisco indique que la vulnérabilité CVE-2026-20245 est activement exploitée.
Lien : https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-AVI-0699/
Date : Fri, 05 Jun 2026 00:00:00 +0000
Titre : Multiples vulnérabilités dans les produits SAP (09 juin 2026)
Description: De multiples vulnérabilités ont été découve

In [2]:
urls_json = []
res=[]
for entry in rss_feed.entries:
    urls_json.append(entry.link+"json/")
for url in urls_json:
    response = requests.get(url)

    if response.status_code == 404:
        print(f"Pas de JSON pour : {url}")
        continue

    if response.status_code != 200:
        print(f"Erreur {response.status_code} pour : {url}")
        continue

    if not response.text.strip():
        print(f"Réponse vide pour : {url}")
        continue

    try:
        data = response.json()
    except Exception as e:
        print(f"JSON invalide pour {url} : {e}")
        continue

    response = requests.get(url)
    data = response.json()
    #Extraction des CVE reference dans la clé cves du dict data
    ref_cves = data.get("cves", [])
    #attention il s’agit d’une liste des dictionnaires avec name et url comme clés
    print( "CVE référencés ", ref_cves)
    # Extraction des CVE avec une regex
    cve_pattern = r"CVE-\d{4}-\d{4,7}"
    cve_list = set(re.findall(cve_pattern, str(data)))
    for cve in cve_list:
         print(f"  - {cve}")
    res.append(cve_list)
print(res)


CVE référencés  [{'name': 'CVE-2026-42897', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-42897'}]
  - CVE-2026-42897
CVE référencés  [{'name': 'CVE-2026-20245', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-20245'}]
  - CVE-2026-20245
CVE référencés  [{'name': 'CVE-2026-44746', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-44746'}, {'name': 'CVE-2026-44750', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-44750'}, {'name': 'CVE-2026-44743', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-44743'}, {'name': 'CVE-2026-44755', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-44755'}, {'name': 'CVE-2026-44754', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-44754'}, {'name': 'CVE-2026-29145', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-29145'}, {'name': 'CVE-2026-44751', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-44751'}, {'name': 'CVE-2026-44757', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-44757'}, {'name': 'CVE-2026-27671', 'url': 'https://www.cve.

In [3]:
import time

def get_cvss_cwe(cve_id):
    try:
        url = f"https://cveawg.mitre.org/api/cve/{cve_id}"
        response = requests.get(url, timeout=10)
        if response.status_code != 200:
            return None, None, None
        data = response.json()
        cna = data["containers"]["cna"]

        try:
            description = cna["descriptions"][0]["value"]
        except:
            description = "Non disponible"

        cvss_score = None
        try:
            metrics = cna.get("metrics", [{}])[0]
            for key in ["cvssV3_1", "cvssV3_0", "cvssV2_0"]:
                if key in metrics:
                    cvss_score = metrics[key]["baseScore"]
                    break
        except:
            pass

        cwe = "Non disponible"
        try:
            problemtype = cna.get("problemTypes", [])
            if problemtype and "descriptions" in problemtype[0]:
                cwe = problemtype[0]["descriptions"][0].get("cweId", "Non disponible")
        except:
            pass

        return description, cvss_score, cwe

    except Exception as e:
        print(f"Erreur MITRE pour {cve_id} : {e}")
        return None, None, None


def get_epss(cve_id):
    try:
        url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
        response = requests.get(url, timeout=10)
        data = response.json()
        epss_data = data.get("data", [])
        if epss_data:
            return float(epss_data[0]["epss"])
        return None
    except Exception as e:
        print(f"Erreur EPSS pour {cve_id} : {e}")
        return None


all_cves = set()
for cve_set in res:
    all_cves.update(cve_set)

print(f"Total CVE uniques à enrichir : {len(all_cves)}")

cve_enriched = {}

for cve_id in all_cves:
    print(f"Enrichissement de {cve_id}...")
    description, cvss_score, cwe = get_cvss_cwe(cve_id)
    epss_score = get_epss(cve_id)
    cve_enriched[cve_id] = {
        "description": description,
        "cvss_score": cvss_score,
        "cwe": cwe,
        "epss_score": epss_score
    }
    time.sleep(0.2)

print("Enrichissement terminé.")

Total CVE uniques à enrichir : 1237
Enrichissement de CVE-2026-8571...
Enrichissement de CVE-2026-23151...
Enrichissement de CVE-2026-46011...
Enrichissement de CVE-2026-11131...
Enrichissement de CVE-2026-45604...
Enrichissement de CVE-2026-45839...
Enrichissement de CVE-2026-10011...
Enrichissement de CVE-2026-46058...
Enrichissement de CVE-2026-43958...
Enrichissement de CVE-2025-58181...
Enrichissement de CVE-2026-43013...
Enrichissement de CVE-2026-41003...
Enrichissement de CVE-2026-10004...
Enrichissement de CVE-2026-46196...
Enrichissement de CVE-2026-47292...
Enrichissement de CVE-2026-46109...
Enrichissement de CVE-2026-9110...
Enrichissement de CVE-2026-23146...
Enrichissement de CVE-2026-8566...
Enrichissement de CVE-2026-45643...
Enrichissement de CVE-2026-23093...
Enrichissement de CVE-2026-8509...
Enrichissement de CVE-2026-45597...
Enrichissement de CVE-2026-9752...
Enrichissement de CVE-2026-8546...
Enrichissement de CVE-2026-23011...
Enrichissement de CVE-2026-46075..

In [6]:
import pandas as pd

rows = []

for entry in rss_feed.entries:
    url_json = entry.link + "json/"
    
    try:
        response = requests.get(url_json, timeout=10)
        if response.status_code != 200:
            continue
        data = response.json()
    except:
        continue

    bulletin_id = data.get("id", "Non disponible")
    titre = entry.title.replace("\n", " ").replace("\r", " ")
    date_pub = entry.published
    lien = entry.link
    type_bulletin = "Alerte" if "alerte" in lien.lower() or "ale" in bulletin_id.lower() else "Avis"

    cve_pattern = r"CVE-\d{4}-\d{4,7}"
    cve_list = list(set(re.findall(cve_pattern, str(data))))

    if not cve_list:
        rows.append({
            "ID ANSSI": bulletin_id,
            "Titre": titre,
            "Type": type_bulletin,
            "Date": date_pub,
            "CVE": None,
            "CVSS Score": None,
            "CWE": None,
            "EPSS Score": None,
            "Description": None,
            "Lien": lien
        })
        continue

    for cve_id in cve_list:
        enriched = cve_enriched.get(cve_id, {})
        description = enriched.get("description")
        if description:
            description = description.replace("\n", " ").replace("\r", " ").replace(";", ",")
        rows.append({
            "ID ANSSI": bulletin_id,
            "Titre": titre,
            "Type": type_bulletin,
            "Date": date_pub,
            "CVE": cve_id,
            "CVSS Score": enriched.get("cvss_score"),
            "CWE": enriched.get("cwe"),
            "EPSS Score": enriched.get("epss_score"),
            "Description": description,
            "Lien": lien
        })

df = pd.DataFrame(rows)
print(df.head())
print(df.shape)

df.to_csv("anssi_cve_consolidated.csv", index=False, sep=";")
print("Fichier CSV exporté.")

         ID ANSSI                                              Titre    Type  \
0  Non disponible  [Màj] Vulnérabilité dans Microsoft Exchange Se...  Alerte   
1  Non disponible  Vulnérabilité dans Cisco Catalyst SD-WAN (05 j...    Avis   
2  Non disponible  Multiples vulnérabilités dans les produits SAP...    Avis   
3  Non disponible  Multiples vulnérabilités dans les produits SAP...    Avis   
4  Non disponible  Multiples vulnérabilités dans les produits SAP...    Avis   

                              Date             CVE  CVSS Score      CWE  \
0  Fri, 15 May 2026 00:00:00 +0000  CVE-2026-42897         8.1   CWE-79   
1  Fri, 05 Jun 2026 00:00:00 +0000  CVE-2026-20245         7.8  CWE-116   
2  Tue, 09 Jun 2026 00:00:00 +0000  CVE-2026-44755         4.3  CWE-346   
3  Tue, 09 Jun 2026 00:00:00 +0000  CVE-2026-44751         7.1  CWE-862   
4  Tue, 09 Jun 2026 00:00:00 +0000  CVE-2026-24315         4.2   CWE-35   

   EPSS Score                                        Description  \
